# Part 3: Data Analysis

Set up modes for dev & testing
- `test`: use small test data set to test BLS-related Spark queries
- `local`: load locally downloaded files and run Spark queries
- `prod`: load files from S3 and run Spark queries

In [ ]:
mode = 'prod' # 'test', 'local', 'prod'

In [ ]:
if mode == 'test':
    min_year = 1995
    max_year = 2001
else:
    min_year = 2013
    max_year = 2018

In [ ]:
import pyspark
from pyspark.sql import SparkSession

# Create a Spark session
spark = (SparkSession.builder 
    .appName("RearcDemo")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") 
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.1") 
    .getOrCreate())

spark.sparkContext.setLogLevel('ERROR')

In [ ]:
if mode == 'prod':
    path = 's3a://rearc-demo-dk/inbound/2026-04-09/pr.data.0.Current'
    path2 = 's3a://rearc-demo-dk/inbound/datausa_population.json'
else:
    path = 'pr.data.0.Current'
    path2 = 'datausa_population.json'
print(f'using paths =\n{path}\n{path2}')

## Part 0
load population and BLS datasets

In [ ]:
if mode == 'test':
    data = [
        ('PRS30006011',	1995,   'Q01',  1),
        ('PRS30006011',	1995,	'Q02',	2),
        ('PRS30006011',	1996,	'Q01',	3),
        ('PRS30006011',	1996,	'Q02',	4),
        ('PRS30006012',	2000,	'Q01',	0),
        ('PRS30006012',	2000,	'Q02',	8),
        ('PRS30006012',	2001,	'Q01',	2),
        ('PRS30006012',	2001,	'Q02',	3)]
    columns = ['series_id','year','period','value']
    bls_df = spark.createDataFrame(data, schema=columns)
else:
    bls_df = (spark.read.format('csv')
        .option('header',True)
        .option('inferSchema', True)
        .option('sep','\t')
        .load(path))
    bls_df.write.csv('out.csv')

Create a Dataframe with current population data. Some unpacking, reformatting of data in required here since the source has a list of structs.

In [ ]:
datausa_df = (spark.read.format('json').load(path2).select('data'))
popdata = [r.data for r in datausa_df.select('data').collect()][0]
popdata_df = spark.createDataFrame(popdata)

Make sure data looks right. Display as a Pandas dataframe

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
bls_df.toPandas()

In [ ]:
popdata_df.toPandas()

## Part 1
Using the dataframe from the population data API (Part 2), generate the mean and the standard deviation of the annual US population across the years [2013, 2018] inclusive.

First, calculate the total population in that range as a sum of the total populations across states.

In [ ]:
from pyspark.sql import functions as F
pop_df = (popdata_df
    .filter((popdata_df.Year >= min_year) & (popdata_df.Year <= max_year))
    .groupBy('Year')
    .agg(F.sum('Population').alias('Population'))
    .withColumnRenamed('Year','year')
    .orderBy('year'))
pop_df.toPandas()

Here is the mean & stddev of the US population across 2013 to 2018. Using the handy Spark dataframe function 'summary'

In [ ]:
pop_df.select('Population').summary("mean", "stddev").show()

## Part 2

Using the dataframe from the time-series (Part 1), For every series_id, find the best year: the year with the max/largest sum of "value" for all quarters in that year. Generate a report with each series id, the best year for that series, and the summed value for that year.

In [ ]:
bls_df.printSchema()
# strip whitespace in column name and reassign
new_cols = [x.strip() for x in bls_df.columns]
bls_renamed = bls_df.toDF(*new_cols)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, desc, regexp_replace, trim
from pyspark.sql import Window

# Step 1: sum `value` across all quarters for each (series_id, year)
yearly_totals = (
    bls_renamed
        .filter((bls_renamed.year >= min_year) & (bls_renamed.year <= max_year))
        .groupBy('series_id', 'year')
        .agg(F.sum('value').alias('value'))
)

# Step 2: rank years within each series_id by total_value (descending)
w = Window.partitionBy('series_id').orderBy(F.col('value').desc())
grouped_df = (
    yearly_totals
    .withColumn('n', F.row_number().over(w))
    .filter(F.col('n') == 1)
    .select(
        'series_id',
        'year',
        'value'
    )
    .orderBy('series_id',desc('value'))
)

In [ ]:
grouped_df.toPandas()

## Part 3

Using both dataframes from Part 1 and Part 2, generate a report that will provide the value for and the population for that given year (if available in the population dataset). The below table shows an example of one row that might appear in the resulting table.

Steps:
- Filter for `series_id == PRS30006032` and `period == Q01`
- drop `footnote_codes` column
- trim whitespace (more efficiet) on join cols
- Join `popdata_df` and `bls_renamed` on `year`
- Sort by year (for ease of use)
- There is some sort of non-printing character in the `series_id`,
  - filtering for == doesn't return anything
  - using `like('PRS30006032%')` returns a list of values


In [ ]:
pop_df = pop_df.withColumn('year', trim('year'))
joined_df = (bls_renamed
    .drop('footnote_codes')
    .withColumn('series_id', trim('series_id'))
    .withColumn('period', trim('period'))
    .withColumn('year', trim('year'))
    .withColumn('value', trim('value'))
    .filter((bls_renamed.series_id.like('PRS30006032%')) & ( bls_renamed.period == 'Q01'))
    .join(pop_df, 'year', 'inner')
    .orderBy('year'))

In [ ]:
joined_df.show(joined_df.count(), truncate=False)